# RQ2: generalisation of UMDA4GI and the three baselines

**Definitions** (the same in every notebook, see the docstring of `analysis.py`):

- *improvement*: % change in CPU cycles of the returned patch on the validation inputs (negative = faster).
  A patch that fails the correctness oracle on the validation inputs counts as 0% (the original program is kept).
- *gap*: for each run, validation change minus training change, in percentage points (positive = the patch
  generalises worse than it trained). Aggregated with the median of the runs.
- *cell*: (software, algorithm, budget), 15 runs = 5 folds × 3 seeds. Each run has 1000 evaluations; the budgets
  100, 250, 500 and 750 are checkpoints of the same run.
- *ranking*: per software, the median of each cell, averaged over the five budgets and ranked; rank sum over the
  four systems (lower = better).

The gap is computed per run and then aggregated (Reviewer 1, comment 5): the median gap is not the difference between the median validation and the median training change.

In [1]:
import pathlib
import sys

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(pathlib.Path('..', 'revision_analysis').resolve()))
import analysis as A

RESULTS = 'results.csv'           # one row per run and budget (checkpoint)
OUT = pathlib.Path('analysis')    # tables written by analysis.py
REFRESH = False                   # True: recompute every table from results.csv
if REFRESH or not (OUT / 'cells.csv').exists():
    A.run(RESULTS, OUT, 'zero')
df = A.load(RESULTS, 'zero')
cells = pd.read_csv(OUT / 'cells.csv')
SW = {'minisat_hack': 'MiniSAT-hack', 'sat4j': 'Sat4J', 'weka': 'Weka', 'pngOptim': 'OptiPNG'}
pd.set_option('display.precision', 2)
pd.set_option('display.max_rows', 300)
pd.set_option('display.width', 200)

## Median gap per software, algorithm and budget

Median of the 15 runs with its bootstrap 95% confidence interval (2000 resamples).

In [2]:
c = cells[cells.group.isin(A.BASE)].copy()
c['value'] = c.apply(lambda r: f"{r.gap_median:.2f} [{r.gap_ci_low:.2f}, {r.gap_ci_high:.2f}]", axis=1)
display(c.pivot_table(index=['software', 'group'], columns='budget', values='value', aggfunc='first'))

budget                             100                   250                   500                    750                    1000
software     group                                                                                                               
minisat_hack FI     2.12 [-0.55, 20.39]  21.76 [-2.90, 29.52]   23.09 [2.70, 45.07]    24.48 [2.70, 51.93]    27.30 [8.00, 54.42]
             GP      9.99 [1.82, 29.83]   28.93 [2.04, 48.01]   24.17 [5.62, 47.69]    38.64 [4.66, 54.67]    28.99 [4.73, 52.77]
             RS     20.77 [0.99, 46.51]  21.96 [13.31, 51.93]  37.87 [20.77, 71.71]  31.53 [-13.97, 61.48]  31.53 [-11.39, 61.48]
             UMDA   2.87 [-1.07, 19.24]    9.56 [1.76, 19.22]    9.56 [0.61, 49.12]    12.08 [0.86, 20.49]   21.23 [-0.57, 48.78]
pngOptim     FI       0.06 [0.02, 0.22]     0.06 [0.05, 0.40]     0.12 [0.01, 0.40]      0.12 [0.01, 0.28]      0.02 [0.01, 0.26]
             GP       0.15 [0.08, 0.16]     0.15 [0.11, 0.17]     0.16 [0.12, 0.31]      0.10 [0.07, 0.25]      0.13 [0.07, 0.38]
             RS       0.12 [0.05, 0.41]     0.30 [0.16, 0.55]     0.26 [0.07, 0.42]      0.26 [0.12, 0.28]      0.23 [0.05, 0.28]
             UMDA     0.07 [0.00, 0.16]    0.00 [-0.08, 0.12]    0.00 [-0.08, 0.12]     0.00 [-0.10, 0.12]    -0.03 [-0.22, 0.08]
sat4j        FI      6.75 [2.75, 14.00]   11.78 [6.75, 16.17]   11.78 [6.75, 16.17]    15.27 [6.75, 20.08]    16.17 [8.29, 21.97]
             GP       3.30 [1.74, 3.94]    3.84 [3.37, 19.79]   3.57 [-0.64, 19.79]     3.80 [2.02, 14.69]     7.37 [1.40, 19.63]
             RS      3.02 [-1.05, 5.19]    3.62 [0.13, 11.37]   11.37 [5.48, 19.86]    11.37 [5.48, 20.91]    11.37 [5.48, 20.91]
             UMDA     5.64 [2.46, 7.72]     5.64 [2.46, 7.52]    5.90 [2.96, 11.09]     9.91 [5.64, 12.11]     9.91 [5.89, 16.47]
weka         FI       1.97 [0.12, 3.54]     3.58 [1.79, 4.46]     2.16 [0.97, 3.58]      1.80 [0.97, 3.69]      2.16 [0.97, 3.69]
             GP      1.74 [-0.70, 2.20]     2.20 [1.74, 3.63]     2.46 [1.78, 3.77]      2.46 [1.92, 3.77]      2.47 [1.74, 4.43]
             RS      0.09 [-0.70, 2.28]    1.05 [-0.70, 2.92]    0.19 [-0.60, 2.92]     0.19 [-0.70, 2.92]     0.19 [-0.70, 1.93]
             UMDA     1.25 [0.09, 3.67]     1.25 [0.44, 3.23]     0.87 [0.09, 2.61]      0.87 [0.00, 2.61]     0.44 [-0.37, 3.93]

## Friedman test per software and budget

Over the four algorithms, with the 15 (fold, seed) pairs as blocks.

In [3]:
fr = pd.read_csv(OUT / 'rq1_rq2_friedman.csv')
fr = fr[fr.metric == 'gap']
display(fr.pivot(index='software', columns='budget', values='p'))
print('significant (p < 0.05):', (fr.p < 0.05).sum(), 'of', len(fr))

budget,100,250,500,750,1000
software,,,,,
minisat_hack,0.33,0.09,2.53e-02,0.24,0.27
pngOptim,0.72,0.02,7.81e-03,0.02,0.01
sat4j,0.21,0.27,8.40e-01,0.49,0.58
weka,0.43,0.14,1.60e-01,0.33,0.88


significant (p < 0.05): 5 of 20


## Pairwise comparisons

Wilcoxon signed-rank tests paired by fold and seed, Holm's correction within each (software, budget, metric), i.e. over its 6 pairs. `a12_a_better` is the probability that a run of `group_a` gives a lower (better) value than a run of `group_b`.

In [4]:
pw = pd.read_csv(OUT / 'rq1_rq2_pairwise.csv')
p = pw[pw.metric == 'gap']
display(p[p.p_holm < 0.05])
print('significant after Holm within each family:', (p.p_holm < 0.05).sum(), 'of', len(p))

,software,budget,group_a,group_b,metric,blocks,median_a,median_b,a12_a_better,p,p_holm
34,minisat_hack,500,RS,UMDA,gap,15,37.87,9.56,0.30,4.27e-03,2.14e-02
35,minisat_hack,500,GP,UMDA,gap,15,24.17,9.56,0.41,8.54e-04,5.13e-03
82,pngOptim,250,RS,UMDA,gap,15,0.30,0.00,0.14,1.53e-03,9.16e-03
94,pngOptim,500,RS,UMDA,gap,15,0.26,0.00,0.17,1.16e-03,6.96e-03
95,pngOptim,500,GP,UMDA,gap,15,0.16,0.00,0.16,1.16e-03,6.96e-03
106,pngOptim,750,RS,UMDA,gap,15,0.26,0.00,0.15,8.54e-04,5.13e-03
107,pngOptim,750,GP,UMDA,gap,15,0.10,0.00,0.28,4.27e-03,2.14e-02
118,pngOptim,1000,RS,UMDA,gap,15,0.23,-0.03,0.21,2.38e-03,1.19e-02
119,pngOptim,1000,GP,UMDA,gap,15,0.13,-0.03,0.20,6.10e-04,3.66e-03


significant after Holm within each family: 9 of 120


## Ranking

In [5]:
display(pd.read_csv(OUT / 'rq2_gap_ranking.csv'))

,group,minisat_hack_value,pngOptim_value,sat4j_value,weka_value,minisat_hack_rank,pngOptim_rank,sat4j_rank,weka_rank,rank_sum
0,UMDA,11.06,8.06e-03,7.40,0.94,1.0,1.0,2.0,2.0,6.0
1,GP,26.14,1.36e-01,4.38,2.26,3.0,3.0,1.0,3.0,10.0
2,FI,19.75,7.52e-02,12.35,2.33,2.0,2.0,4.0,4.0,12.0
3,RS,28.73,2.37e-01,8.15,0.34,4.0,4.0,3.0,1.0,12.0


### Sensitivity: the same ranking with the mean instead of the median

The median is used because a few extreme runs would dominate the mean (see below); the tests are rank-based and do not depend on this choice.

In [6]:
means = (df[df.group.isin(A.BASE)].groupby(['software', 'group', 'checkpoint'])['gap'].mean()
         .rename('gap_mean').reset_index())
display(A.ranking(means, A.BASE, 'gap_mean'))

software,group,minisat_hack_value,pngOptim_value,sat4j_value,weka_value,minisat_hack_rank,pngOptim_rank,sat4j_rank,weka_rank,rank_sum
0,UMDA,15.60,-0.49,10.11,1.27,1.0,2.0,1.0,2.0,6.0
1,GP,28.70,0.19,10.68,2.33,2.0,3.0,2.0,4.0,11.0
2,RS,33.98,0.21,11.10,1.14,3.0,4.0,3.0,1.0,11.0
3,FI,47.63,-0.51,12.42,2.29,4.0,1.0,4.0,3.0,12.0


## Patch length

Mean number of edits of the returned patch (0 when the run found no improving patch).

In [7]:
b = df[df.group.isin(A.BASE)]
display(pd.DataFrame({'all budgets': b.groupby('group').patch_length.mean(),
                      '1000 evaluations': b[b.checkpoint == 1000].groupby('group').patch_length.mean()}))

,all budgets,1000 evaluations
group,,
FI,5.06,6.68
GP,2.63,3.87
RS,1.00,1.00
UMDA,1.63,1.67


## Patch length and gap

Spearman correlation per software over every run and budget of the four algorithms, leaving out runs without a patch. See `Ablations.ipynb` for the length-controlled comparison.

In [8]:
from scipy.stats import spearmanr
rows = []
for sw, d in b[b.patch_length > 0].dropna(subset=['gap']).groupby('software'):
    rho, p = spearmanr(d.patch_length, d.gap)
    rows.append({'software': sw, 'n': len(d), 'rho': rho, 'p': p})
display(pd.DataFrame(rows))

,software,n,rho,p
0,minisat_hack,300,-0.05,4.32e-01
1,pngOptim,292,-0.04,4.52e-01
2,sat4j,300,0.13,2.74e-02
3,weka,294,0.22,1.32e-04
